GitHub: https://github.com/lucidrains/vit-pytorch

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
import torchvision
from torchvision import datasets, transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

In [ ]:
!pip install vit-pytorch

In [ ]:
train = pd.read_csv("../input/cassava-leaf-disease-classification/train.csv")
sub = pd.read_csv("../input/cassava-leaf-disease-classification/sample_submission.csv")

# Hyperparams

In [ ]:
NUM_CLASSES = train["label"].nunique()
TARGET_SIZE = (320, 320)
IMG_SIZE = 320
EPOCHS = 25
VALIDATION_SPLIT = 0.15
BATCH_SIZE = 32
LR = 2e-5
GAMMA = 0.7

In [ ]:
train_transforms = transforms.Compose(
    [
        transforms.Resize(TARGET_SIZE),
        transforms.RandomResizedCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ]
)

val_transforms = transforms.Compose(
    [
        transforms.Resize(TARGET_SIZE),
        transforms.ToTensor(),
    ]
)

test_transforms = transforms.Compose(
    [
        transforms.Resize(TARGET_SIZE),
        transforms.ToTensor(),
    ]
)

In [ ]:
train_list, valid_list = train_test_split(train, 
                                          test_size=0.15,
                                          stratify=train["label"],
                                          random_state=2987346)

In [ ]:

class MyDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, transform = None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        img = (Image.open("../input/cassava-leaf-disease-classification/train_images/" + row["image_id"]))
        label = row["label"]
        if not self.transform:
            return torchvision.transforms.functional.to_tensor(img), label
        img_transformed = self.transform(img)
        return img_transformed, label


train_dataset = MyDataset(train_list, train_transforms)
val_dataset = MyDataset(valid_list, val_transforms)


class TestDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, transform = None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        img = (Image.open("../input/cassava-leaf-disease-classification/test_images/" + row["image_id"]))
        if not self.transform:
            return torchvision.transforms.functional.to_tensor(img), label
        img_transformed = self.transform(img)
        return img_transformed
    
test_dataset = TestDataset(sub, test_transforms)

In [ ]:
train_loader = DataLoader(dataset = train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(dataset = val_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset = test_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
print(len(train_dataset), len(train_loader))
print(len(val_dataset), len(val_loader))

In [ ]:
device = ('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from vit_pytorch import ViT


model = ViT(
    image_size = IMG_SIZE,
    patch_size = 16,
    num_classes = NUM_CLASSES,
    dim = 1024,
    depth = 6,
    heads = 8,
    mlp_dim = 2048,
    dropout = 0.1,
    emb_dropout = 0.1
).to(device)

In [ ]:
# loss function
criterion = nn.CrossEntropyLoss()
# optimizer
optimizer = optim.Adam(model.parameters(), lr=LR)
# scheduler
scheduler = StepLR(optimizer, step_size=1, gamma=GAMMA)

In [ ]:
from tqdm.notebook import tqdm

# Training

In [ ]:
for epoch in range(EPOCHS):
    epoch_loss = 0
    epoch_accuracy = 0

    for data, label in tqdm(train_loader):
        data = data.to(device)
        label = label.to(device)

        output = model(data)
        loss = criterion(output, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        acc = (output.argmax(dim=1) == label).float().mean()
        epoch_accuracy += acc / len(train_loader)
        epoch_loss += loss / len(train_loader)

    with torch.no_grad():
        epoch_val_accuracy = 0
        epoch_val_loss = 0
        for data, label in val_loader:
            data = data.to(device)
            label = label.to(device)

            val_output = model(data)
            val_loss = criterion(val_output, label)

            acc = (val_output.argmax(dim=1) == label).float().mean()
            epoch_val_accuracy += acc / len(val_loader)
            epoch_val_loss += val_loss / len(val_loader)

    print(
        f"Epoch : {epoch+1} - loss : {epoch_loss:.4f} - acc: {epoch_accuracy:.4f} - val_loss : {epoch_val_loss:.4f} - val_acc: {epoch_val_accuracy:.4f}\n"
    )

# Make predictions

In [ ]:
import numpy as np

model.eval()
preds = []

for data in test_loader:
    inputs = data.to(device)

    with torch.no_grad():
        outputs = model(inputs)

    preds.append(outputs.sigmoid().detach().cpu().numpy())

preds = np.concatenate(preds)

In [ ]:
predictions = np.argmax(preds, axis=1)
predictions

In [ ]:
sub["label"] = predictions
sub.to_csv("submission.csv", index=False)